# Notebook 2 — Feature Engineering (Distributed NLP Pipeline)

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Pipeline stages
```
raw text → Tokenizer → StopWordsRemover → HashingTF → IDF → feature vector
```

All transformations are **Spark MLlib Transformers** — they run distributed
across executors, not on the driver, which is what separates this from a
scikit-learn `TfidfVectorizer` (single-node, RAM-bound).

In [ ]:
# ── 2.1  Bootstrap ──────────────────────────────────────────────────────
import os, sys

os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# ── 2.2  Load Parquet produced by Notebook 1 ────────────────────────────
PARQUET_PATH = "../data/parquet/news_articles"

df = spark.read.parquet(PARQUET_PATH)
print(f"Loaded {df.count():,} rows  |  Partitions: {df.rdd.getNumPartitions()}")
df.printSchema()
df.groupBy("label").count().show()

## 2.3  Label Strategy

The dataset generated in Notebook 1 already contains labels:
- **label = 0** → Reliable news articles  
- **label = 1** → Fake / unreliable news articles  

In a production Common Crawl pipeline, labels would come from one of:
- **Option A — Domain-based weak labelling:** Map URLs to curated lists of reliable vs. unreliable domains (MediaBiasFactCheck, OpenSources).
- **Option B — Supervised join:** Cross-reference URLs against labelled datasets like FakeNewsNet, LIAR, or ISOT.

Since our data is pre-labelled, we proceed directly to text preprocessing.

In [ ]:
# ── 2.4  Text preprocessing (no UDFs — all Spark built-in functions) ─────
from pyspark.sql.functions import regexp_replace, trim, length, lower, col

labelled_df = (
    df
    # Remove URLs embedded in text
    .withColumn("text", regexp_replace("text", r"https?://\S+", ""))
    # Remove non-alphabetic characters (keep spaces)
    .withColumn("text", regexp_replace("text", r"[^a-zA-Z\s]", ""))
    # Collapse multiple spaces
    .withColumn("text", regexp_replace("text", r"\s+", " "))
    # Trim & lowercase
    .withColumn("text", trim(lower(col("text"))))
    # Drop if text too short after cleaning
    .filter(length("text") >= 100)
)

print(f"After text cleaning: {labelled_df.count():,} rows")
labelled_df.select("text").show(2, truncate=120)

In [ ]:
# ── 2.5  Verify text preprocessing ──────────────────────────────────────
print("Label distribution after cleaning:")
labelled_df.groupBy("label").count().show()
print(f"Sample cleaned text (first 200 chars):")
labelled_df.select("text").show(3, truncate=200)

In [ ]:
# ── 2.6  MLlib Feature Pipeline ─────────────────────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

# Stage 1: Tokenizer — splits text into word array
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Stage 2: StopWordsRemover — removes English stop words
#   WHY: stop words ("the", "is", "and") add noise and inflate dimensionality
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# Stage 3: HashingTF — maps words to fixed-size feature vector
#   numFeatures=2**14 (16384) — balances collision rate vs memory.
#   Kept at 2^14 to avoid OOM during Random Forest cross-validation.
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=2**14)

# Stage 4: IDF — down-weight common terms, up-weight rare terms
#   minDocFreq=5 filters noise from very rare typos/tokens
idf = IDF(inputCol="raw_features", outputCol="features", minDocFreq=5)

feature_pipeline = Pipeline(stages=[tokenizer, stopwords_remover, hashing_tf, idf])

print("Feature pipeline stages:")
for i, stage in enumerate(feature_pipeline.getStages()):
    print(f"  {i}: {type(stage).__name__}")

In [ ]:
# ── 2.7  Fit pipeline & transform ───────────────────────────────────────
labelled_df.persist(StorageLevel.MEMORY_AND_DISK)

pipeline_model = feature_pipeline.fit(labelled_df)
features_df = pipeline_model.transform(labelled_df)

# Select only columns needed downstream
features_df = features_df.select("title", "subject", "date", "source", "label", "features")

features_df.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Feature vectors: {features_df.count():,} rows")
features_df.show(3, truncate=60)

labelled_df.unpersist()

In [ ]:
# ── 2.8  Train / Validation / Test Split with Stratification ────────────
# Stratified split: maintain label ratio across all splits
# Spark doesn't have sklearn's StratifiedKFold, so we use sampleBy

fractions = {0: 0.7, 1: 0.7}   # 70% of each class for train
train_df = features_df.stat.sampleBy("label", fractions, seed=42)
remaining = features_df.subtract(train_df)

fractions_vt = {0: 0.5, 1: 0.5}  # 50/50 split of remaining → ~15% val, ~15% test
val_df  = remaining.stat.sampleBy("label", fractions_vt, seed=42)
test_df = remaining.subtract(val_df)

for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    total = split_df.count()
    pos   = split_df.filter(col("label") == 1).count()
    print(f"{name:6s}: {total:>8,} rows  |  label=1 ratio: {pos/max(total,1):.3f}")

In [ ]:
# ── 2.9  Save feature-engineered splits to Parquet ──────────────────────
FEATURES_BASE = "../data/parquet/features"

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    path = f"{FEATURES_BASE}/{name}"
    split_df.write.mode("overwrite").parquet(path)
    print(f"✓ {name} → {path}")

# Save fitted pipeline model for reproducibility
pipeline_model.write().overwrite().save("../data/models/feature_pipeline")
print("✓ Feature pipeline model saved")

In [ ]:
# ── 2.10  Class distribution for Tableau export ─────────────────────────
import pandas as pd

class_dist = (
    features_df
    .groupBy("label")
    .agg(F.count("*").alias("count"))
    .toPandas()
)
class_dist["label_name"] = class_dist["label"].map({0: "Reliable", 1: "Unreliable"})
class_dist.to_csv("../tableau/class_distribution.csv", index=False)
print("✓ Exported class_distribution.csv for Tableau")
class_dist

In [ ]:
# ── Cleanup ──
features_df.unpersist()
# spark.stop()